# MCDM Weight Sensitivity — CO₂ Storage Sites

## Objective of this analysis

**The purpose of this One-at-a-Time (OAT) sensitivity analysis is to assess how robust a multi-criteria ranking is to changes in criterion weights.** In particular, it asks: *Would we still recommend the same CO₂ storage site if the decision-maker placed more or less emphasis on storage capacity, injectivity or storage cost?*

The analysis:
1. Calculates a **baseline ranking** for three illustrative sites using criterion scores (1–5) and a weighted additive model.
2. **Varies one selected criterion weight** from 0% to 100% while proportionally redistributing the remaining weights so their total stays at 100%.
3. Recalculates the overall scores and examines **where site rankings or first-place alternatives change**.
4. Uses sensitivity curves, a robustness summary and a **first-place stability map** to identify the weights at which the preferred alternative is no longer the same.

**How to interpret the result:** A site that remains first across a specified set of plausible weights is *robust within those assumptions*; a site that loses first place after small changes in weights is *sensitive to decision-maker preferences*. The percentages in the robustness plots describe the tested weight grid; **they are not probabilities of geological outcomes**.

**What the turquoise “Tie” color means in the stability map:** At that tested weight, **two or more sites have the same highest total weighted score**. Neither is uniquely preferred by this model. The colored area represents a **sampled grid cell**, not a continuous interval of equality or uncertainty. With the default example and 5-percentage-point steps, capacity = **50%** produces an A–B tie, and injectivity = **25%** produces a B–C tie. A tie between two *lower-ranked* sites would not be shown in this winner map.

**Reference and scope:** This fictional three-site example illustrates the principles discussed in Marco Dean (2022), *A Practical Guide to Multi-Criteria Analysis*, **Section 5.8, pp. 69–71**, especially the hypothetical weight-sensitivity example in **Figure 31 (p. 70)**. The site scores, weight scenarios, figures and specific numerical results here were developed for this notebook, **not reproduced from Dean**.

**Run in Colab:** Choose **Runtime → Run all**. Code is collapsed by default; choose *Show code* to inspect it. Edit the inputs in Section 1 and rerun to update all tables and figures.

In [ ]:
#@title Import libraries { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


## 1. Input tables

**Table 1 (site performance):** Each fictional site has scores from 1 (worst) to 5 (best). *Storage Cost* is a **desirability** score: 5 means lower actual cost, not a monetary amount.

**Table 2 (weights):** Baseline preferences sum to 100%: capacity 40%, injectivity 35%, cost 25%.

**OAT protocol:** Vary **one designated weight** independently from 0% to 100%; adjust the other two weights *proportionally* to preserve their ratio. For example, capacity = 0% implies injectivity = 58.33%, cost = 41.67%; capacity = 40% implies 35% and 25%. There are three separate OAT experiments, not one.

In [ ]:
#@title Editable inputs { display-mode: "form" }
capacity_weight_pct = 40 #@param {type:"number"}
injectivity_weight_pct = 35 #@param {type:"number"}
cost_weight_pct = 25 #@param {type:"number"}
sensitivity_step_pct = 5 #@param {type:"number"}

# Fictional scores: each row is a site; each column is a criterion.
# You can replace these scores with validated 1–5 desirability scores.
site_scores = pd.DataFrame(
    {
        "Storage capacity": [5, 3, 4],
        "Injectivity": [3, 5, 2],
        "Storage cost": [2, 4, 5],
    },
    index=["Site A", "Site B", "Site C"],
)
criteria = list(site_scores.columns)
weights_pct = np.array([capacity_weight_pct, injectivity_weight_pct, cost_weight_pct], dtype=float)
if np.any(~np.isfinite(weights_pct)) or np.any(weights_pct < 0) or not np.isclose(weights_pct.sum(), 100):
    raise ValueError("Baseline weights must be finite, nonnegative, and sum to 100%.")
numeric_scores = site_scores.to_numpy(dtype=float)
if not np.all(np.isfinite(numeric_scores)) or not np.all((numeric_scores >= 1) & (numeric_scores <= 5)):
    raise ValueError("All scores must be numeric and between 1 and 5.")
if sensitivity_step_pct <= 0 or sensitivity_step_pct > 100 or not np.isclose(100 / sensitivity_step_pct, round(100 / sensitivity_step_pct)):
    raise ValueError("Sensitivity step must divide 100 exactly (e.g. 1, 2, 5, 10).")
weights = weights_pct / 100
display(Markdown("**Table 1. Fictional site-performance scores on a 1–5 desirability scale.**"))
display(site_scores)
display(Markdown("*Note: 5 indicates highest desirability; for Storage Cost, 5 represents lower cost.*"))
display(Markdown("**Table 2. Baseline weights assigned to each evaluation criterion.**"))
display(pd.DataFrame({"Criterion": criteria, "Weight (%)": weights_pct}))
display(Markdown("*Note: criterion weights must total 100%.*"))
print("Editable inputs validated successfully.")


## 2. Table 3 / Figure 1 — baseline ranking

Weighted sum: $S_i = \sum_j w_j s_{ij}$. **Table 3** shows scores and ranks; **Figure 1** shows the same total scores as bars. The scores are dimensionless desirability indices, **not physical CO₂ capacity or actual costs**.

**With the default assumptions:** B = **3.95** (first); A = **3.55** and C = **3.55** (joint second). The A/C tie is meaningful and alphabetical display order must not be interpreted as a preference.

In [ ]:
#@title Baseline ranking { display-mode: "form" }
def weighted_results(score_table, weight_vector):
    if not np.isclose(np.sum(weight_vector), 1) or np.any(weight_vector < -1e-12):
        raise ValueError("Weights must sum to one and be nonnegative.")
    values = score_table.to_numpy(dtype=float) @ np.asarray(weight_vector, dtype=float)
    result = pd.DataFrame({"Site": score_table.index, "Total score": values})
    result["Rank"] = result["Total score"].rank(method="min", ascending=False).astype(int)
    return result.sort_values(["Total score", "Site"], ascending=[False, True]).reset_index(drop=True)

def preference_signature(table):
    """Pairwise order with numerical ties handled explicitly."""
    scores = table.set_index("Site")["Total score"]
    names = sorted(scores.index)
    return tuple(0 if np.isclose(scores[x], scores[y], atol=1e-9, rtol=0)
                 else (1 if scores[x] > scores[y] else -1)
                 for i, x in enumerate(names) for y in names[i+1:])

baseline = weighted_results(site_scores, weights)
display(Markdown("**Table 3. Baseline ranking of the three fictional CO₂ storage sites.**"))
display(baseline.round(3))
display(Markdown("*Note: total scores are dimensionless weighted desirability indices; equal scores share the same rank.*"))
ax = baseline.plot.bar(x="Site", y="Total score", legend=False, figsize=(7, 4))
ax.set_ylabel("Weighted score (1–5)")
ax.set_ylim(0, 5)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(Markdown("**Figure 1.** Baseline weighted desirability scores for the three fictional storage sites under the selected criterion weights. The highest score indicates the preferred site under the assumed model; ties represent equal total desirability."))


## 3. Table 4 — complete OAT sensitivity calculations

**Why the old table showed only Storage Capacity:** `head(10)` displayed only the first 10 of **63 rows** (21 capacity + 21 injectivity + 21 cost). The updated full table below shows all criteria, with Colab's interactive table filtering where available.

Read each row as follows: **Criterion varied** = the experiment; **Tested weight** = chosen weight for that criterion; the three **weight** columns = adjusted 100% allocation; the three **score** columns = recalculated overall scores; **Winner(s)** = highest-scoring alternative(s); **Display order** = formatted rank order, with tied sites displayed alphabetically. **Order changed vs baseline** compares pairwise preferences (including exact ties), not arbitrary alphabetical order.

In [ ]:
#@title Calculate all OAT scenarios { display-mode: "form" }
def rescale_weights(base_weights, index, tested_weight):
    if not 0 <= tested_weight <= 1:
        raise ValueError("Tested weight must be between 0 and 1.")
    original_rest = 1 - base_weights[index]
    if original_rest <= 1e-12:
        if np.isclose(tested_weight, 1):
            result = np.zeros_like(base_weights, dtype=float)
            result[index] = 1
            return result
        raise ValueError("Cannot proportionally rescale other zero baseline weights.")
    result = np.asarray(base_weights, dtype=float) * ((1 - tested_weight) / original_rest)
    result[index] = tested_weight
    return result

rows = []
steps = int(round(100 / sensitivity_step_pct))
baseline_signature = preference_signature(baseline)
for j, criterion in enumerate(criteria):
    for k in range(steps + 1):
        tested_weight = k / steps
        trial_weights = rescale_weights(weights, j, tested_weight)
        trial = weighted_results(site_scores, trial_weights)
        best = trial["Total score"].max()
        winners = sorted(trial.loc[np.isclose(trial["Total score"], best), "Site"].tolist())
        rows.append({
            "Criterion varied": criterion,
            "Tested weight (%)": 100 * tested_weight,
            **{f"{name} weight (%)": 100 * trial_weights[i] for i, name in enumerate(criteria)},
            **{f"{site} score": float(trial.loc[trial["Site"].eq(site), "Total score"].iloc[0]) for site in site_scores.index},
            "Winner(s)": " / ".join(winners),
            "Display order": " > ".join(trial["Site"]),
            "Order changed vs baseline": preference_signature(trial) != baseline_signature,
        })
sensitivity = pd.DataFrame(rows)
display(Markdown("**Table 4. One-at-a-time sensitivity results across all three criteria and tested weights.**"))
try:
    from google.colab import data_table
    display(data_table.DataTable(sensitivity.round(2), include_index=False, num_rows_per_page=15))
except ImportError:
    display(sensitivity.round(2))
display(Markdown("*Note: each row changes one criterion weight, adjusts remaining weights proportionally, and recalculates scores and the preferred alternative.*"))
print(f"Evaluated {len(sensitivity)} scenarios ({len(criteria)} criteria × {steps+1} tested weights).")


## 4. Figure 2 — OAT sensitivity curves

**Methodological source:** Dean (2022), **Section 5.8, pp. 69–70**, especially the hypothetical single-weight test in **Figure 31 (p. 70)**. The three panels below are **original curves from this fictional example**.

**How to read:** Each panel varies *only the named criterion* (x-axis). The y-axis shows the **overall weighted score**, not the score of that individual criterion. Other weights automatically change proportionally. **Blue: A; orange: B; green: C; gray dashed: baseline weight.** Curves crossing means equal total scores. A crossing between two lower-ranked sites need not change the winner.

**Default-data interpretation:**
- **Capacity:** B wins at weights below **50%**, A and B tie at **50%**, then A wins. At baseline 40%, B wins; stronger capacity preference helps A (capacity desirability 5).
- **Injectivity:** C wins below **25%**, C and B tie at **25%**, then B wins. At baseline 35%, B wins; increased injectivity importance helps B (injectivity desirability 5).
- **Cost:** A wins below **6.25%**; B wins from **6.25% to about 46.43%**; C wins above **46.43%**. Baseline 25% favors B; stronger cost preference benefits C (best low-cost desirability 5).

Thresholds are derived from the *default fictional scores and weights*, not supplied by Dean. If you change inputs, use the updated curves to interpret the new results.

In [ ]:
#@title Figure 2 — Sensitivity curves { display-mode: "form" }
fig, axes = plt.subplots(1, len(criteria), figsize=(17, 4.5), sharey=True)
for j, (criterion, ax) in enumerate(zip(criteria, axes)):
    subset = sensitivity.loc[sensitivity["Criterion varied"] == criterion]
    for site in site_scores.index:
        ax.plot(subset["Tested weight (%)"], subset[f"{site} score"], label=site, linewidth=2)
    ax.axvline(weights_pct[j], linestyle="--", color="gray", linewidth=1, label="Baseline")
    ax.set_xlabel(f"{criterion} weight (%)")
    ax.set_xlim(0, 100)
    ax.grid(alpha=.25)
    ax.legend(loc="best", fontsize=9, framealpha=0.9)
axes[0].set_ylabel("Weighted overall score (1–5)")

plt.tight_layout()
plt.show()
display(Markdown("**Figure 2.** One-at-a-time (OAT) weight sensitivity of total weighted desirability scores for Sites A, B and C. In each panel, the indicated criterion weight varies from 0% to 100% while the other two weights are adjusted proportionally. Dashed vertical lines indicate baseline weights; curve intersections identify equal total scores and potential ranking changes. Methodological inspiration: Dean (2022), Section 5.8, Figure 31 (p. 70); the results shown here use our fictional data."))


## 5. Table 5 and Figures 3–4 — robustness and winner map

**Table 5:** Counts sensitivity cases and the percentage of tested weight settings changing the outcome.

**Figure 3 — Meaning of the two bars:**
- **Displayed ranking changes** = at least one pair of sites changes relative order **or changes from a tie into a strict preference**. The best site could remain unchanged. The analysis uses pairwise, tie-aware comparisons, not alphabetical display order.
- **First-place set changes** = the **identity of the winning site or tied winners** differs from the baseline. With the default inputs, the sole baseline winner is B. This metric directly measures whether the preferred site is stable.

The chart's percentages are the fraction of the 21 *tested weights*, **not a probability**. Default results show that the choice is more sensitive to broad variations in capacity and cost weights than injectivity, though the weights 0–100% may not all be plausible. Because A and C are initially tied, the first metric also counts breaking that tie.

**Figure 4 — How to interpret the colored winner map:** This is a **decision-stability diagram, not a geographical map**. Each row is a *separate* OAT experiment; the horizontal axis is the weight of the named criterion. The color identifies the site with the **highest overall weighted score**. A change from one site's color to another means that the recommended site changes.

**The turquoise Tie cell means co-winners:** Two or more alternatives have *equal highest total scores* at that **particular sampled weight**. With default inputs, **capacity = 50%** gives an **A–B tie**, and **injectivity = 25%** gives a **B–C tie**. These tie points mark the boundaries between neighboring winner regions. The turquoise rectangle has a visible width only because the heatmap draws a discrete 5% grid cell: **do not interpret that rectangle as a range of weights for which the sites are continuously tied**. Actual ties between grid points may be missed; consult the curves for those intersections. Lower-ranked ties are not shown in this map.

**Default-data interpretation of the map:** When capacity becomes dominant above 50%, A replaces B. For injectivity below 25%, C replaces B. For cost, A is favored at very low weights, B at intermediate weights, and C above roughly 46.43%. This is consistent with Figure 2's curves.

**Paper-style conclusion:** Site B ranks first under the baseline weighting scheme, but the preferred site changes when the preference structure changes. The result demonstrates **weight dependence rather than a universally best geological site**. These toy results cannot support a real investment decision. The WSM is compensatory, so safety and permitting conditions should be screened separately; real applications also require quality-controlled scores, defensible weight ranges, criterion independence checks and assessment of uncertainty in site data.

In [ ]:
#@title Figures 3–4 — Robustness dashboard { display-mode: "form" }
baseline_best = baseline["Total score"].max()
baseline_winners = " / ".join(sorted(
    baseline.loc[np.isclose(baseline["Total score"], baseline_best), "Site"].tolist()
))
sensitivity["First-place set changed"] = sensitivity["Winner(s)"] != baseline_winners
summary = (
    sensitivity.groupby("Criterion varied", sort=False)
    .agg(
        Tested_scenarios=("Order changed vs baseline", "size"),
        Ranking_changes=("Order changed vs baseline", "sum"),
        First_place_changes=("First-place set changed", "sum"),
    )
    .reset_index()
)
summary["Ranking changes (%)"] = 100 * summary["Ranking_changes"] / summary["Tested_scenarios"]
summary["First-place changes (%)"] = 100 * summary["First_place_changes"] / summary["Tested_scenarios"]
print("Baseline winner(s):", baseline_winners)
display(Markdown("**Table 5. Summary of ranking and first-place sensitivity under one-at-a-time weight variation.**"))
display(summary.round(1))
display(Markdown("*Note: percentages indicate the proportion of tested weight settings that change pairwise ranking or first-place site(s), not probabilities of geological outcomes.*"))

ax = summary.plot.bar(
    x="Criterion varied",
    y=["Ranking changes (%)", "First-place changes (%)"],
    figsize=(9, 4.5),
    rot=0,
)
ax.set_ylabel("Share of tested weights (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.set_title("Sensitivity to each criterion's weight")
ax.legend(["Displayed ranking changes", "First-place set changes"])
plt.tight_layout()
plt.show()
display(Markdown("**Figure 3.** Percentage of tested weight settings that alter the pairwise ranking or change the set of first-place sites, evaluated separately for each criterion. These percentages describe the specified OAT grid rather than likelihoods of real-world outcomes."))

winner_labels = list(site_scores.index) + ["Tie"]
winner_to_code = {name: i for i, name in enumerate(winner_labels)}
winner_matrix = []
for criterion in criteria:
    subset = sensitivity.loc[sensitivity["Criterion varied"] == criterion]
    winner_matrix.append([
        winner_to_code[w] if w in winner_to_code else winner_to_code["Tie"]
        for w in subset["Winner(s)"]
    ])
winner_matrix = np.asarray(winner_matrix)
fig, ax = plt.subplots(figsize=(12, 3.5))
from matplotlib.colors import BoundaryNorm
cmap = plt.get_cmap("tab10", len(winner_labels))
norm = BoundaryNorm(np.arange(-0.5, len(winner_labels) + 0.5, 1), cmap.N)
im = ax.imshow(winner_matrix, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)
ax.set_yticks(range(len(criteria)), criteria)
ax.set_xticks(range(0, steps + 1, max(1, steps // 10)))
ax.set_xticklabels([f"{100 * k / steps:.0f}%" for k in range(0, steps + 1, max(1, steps // 10))])
ax.set_xlabel("Tested criterion weight")

cbar = fig.colorbar(im, ax=ax, ticks=range(len(winner_labels)), pad=0.02)
cbar.ax.set_yticklabels(winner_labels)
plt.tight_layout()
plt.show()
display(Markdown("**Figure 4.** First-place stability map for three independent OAT experiments. Each row varies the named criterion’s weight; each color shows the top-ranked site at a sampled weight. Turquoise **Tie** cells denote co-winners with equal highest overall scores at that specific tested weight (default: A–B at 50% capacity; B–C at 25% injectivity). The colored cell width reflects the 5% sampling interval, **not a continuous tie zone**. A color transition indicates a switch in the preferred site."))


## 6. Citation and limits

**Dean, M. (2022). *A Practical Guide to Multi-Criteria Analysis*. University College London. Section 5.8, pp. 69–71; Figure 31, p. 70; Table 18, p. 71.**

Dean describes examining whether ranking changes with criterion scores and weights, how strongly it changes and which inputs are influential. His Figure 31 is a **hypothetical test** of changing a weight; the numerical scores, charts, crossing points, robustness bars and winner map here are constructed specifically for this teaching example, **not taken from Dean**.

**Scope:** Weight sensitivity only, not probabilistic geological uncertainty. The uniform 0–100% grid is an exploratory design, not a prior probability on weights. Use screening for non-compensatory safety and regulatory requirements before ranking real CO₂ storage sites.